#### Faiss
Facebook AI Similarity Search (Faiss) is a library for efficient similarity search and clustering of dense vectors. It contains algorithms that search in sets of vectors of any size, up to ones that possibly do not fit in RAM. It also contains supporting code for evaluation and parameter tuning.

In [2]:
from langchain_community.document_loaders import TextLoader
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import OllamaEmbeddings
from langchain_text_splitters import CharacterTextSplitter


loader=TextLoader("speech.txt")

documents=loader.load()

text_splitter=CharacterTextSplitter(
    chunk_size=200,
    chunk_overlap=30,
)

docs=text_splitter.split_documents(documents)

Created a chunk of size 470, which is longer than the specified 200
Created a chunk of size 347, which is longer than the specified 200
Created a chunk of size 670, which is longer than the specified 200
Created a chunk of size 984, which is longer than the specified 200
Created a chunk of size 791, which is longer than the specified 200


In [3]:
docs

[Document(metadata={'source': 'speech.txt'}, page_content='The world must be made safe for democracy. Its peace must be planted upon the tested foundations of political liberty. We have no selfish ends to serve. We desire no conquest, no dominion. We seek no indemnities for ourselves, no material compensation for the sacrifices we shall freely make. We are but one of the champions of the rights of mankind. We shall be satisfied when those rights have been made as secure as the faith and the freedom of nations can make them.'),
 Document(metadata={'source': 'speech.txt'}, page_content='Just because we fight without rancor and without selfish object, seeking nothing for ourselves but what we shall wish to share with all free peoples, we shall, I feel confident, conduct our operations as belligerents without passion and ourselves observe with proud punctilio the principles of right and of fair play we profess to be fighting for.'),
 Document(metadata={'source': 'speech.txt'}, page_content

In [5]:
embeddings=OllamaEmbeddings(model="nomic-embed-text")

db=FAISS.from_documents(docs,embeddings)
db

In [8]:
# querrying 
query="What is the speech about?"
docs=db.similarity_search(query)
docs[0].page_content


'It is a distressing and oppressive duty, gentlemen of the Congress, which I have performed in thus addressing you. There are, it may be, many months of fiery trial and sacrifice ahead of us. It is a fearful thing to lead this great peaceful people into war, into the most terrible and disastrous of all wars, civilization itself seeming to be in the balance. But the right is more precious than peace, and we shall fight for the things which we have always carried nearest our heartsâ€”for democracy, for the right of those who submit to authority to have a voice in their own governments, for the rights and liberties of small nations, for a universal dominion of right by such a concert of free peoples as shall bring peace and safety to all nations and make the world itself at last free.'

#### As a Retriever
We can also convert the vectorstore into a Retriever class. This allows us to easily use it in other LangChain methods, which largely work with retrievers

In [11]:
'''Retriever is a component that allows you to retrieve relevant documents 
from a vector store based on a query. In this case, we are using the FAISS 
vector store to perform similarity search. The retriever is created from the 
FAISS database and can be used to invoke queries to get relevant documents.
'''

retriever=db.as_retriever()
docs=retriever.invoke(query)
docs[0].page_content

'It is a distressing and oppressive duty, gentlemen of the Congress, which I have performed in thus addressing you. There are, it may be, many months of fiery trial and sacrifice ahead of us. It is a fearful thing to lead this great peaceful people into war, into the most terrible and disastrous of all wars, civilization itself seeming to be in the balance. But the right is more precious than peace, and we shall fight for the things which we have always carried nearest our heartsâ€”for democracy, for the right of those who submit to authority to have a voice in their own governments, for the rights and liberties of small nations, for a universal dominion of right by such a concert of free peoples as shall bring peace and safety to all nations and make the world itself at last free.'

## Similarity Search with score
There are some FAISS specific methods. One of them is similarity_search_with_score, which allows you to return not only the documents but also the distance score of the query to them. The returned distance score is L2 distance(Manhattan Distance). Therefore, a lower score is better.

In [12]:
docs_and_score=db.similarity_search_with_score(query)
docs_and_score

[(Document(id='034f0cc7-6878-41bf-919c-ce663bdc17ae', metadata={'source': 'speech.txt'}, page_content='It is a distressing and oppressive duty, gentlemen of the Congress, which I have performed in thus addressing you. There are, it may be, many months of fiery trial and sacrifice ahead of us. It is a fearful thing to lead this great peaceful people into war, into the most terrible and disastrous of all wars, civilization itself seeming to be in the balance. But the right is more precious than peace, and we shall fight for the things which we have always carried nearest our heartsâ€”for democracy, for the right of those who submit to authority to have a voice in their own governments, for the rights and liberties of small nations, for a universal dominion of right by such a concert of free peoples as shall bring peace and safety to all nations and make the world itself at last free.'),
  np.float32(408.5108)),
 (Document(id='d0844a28-8c5b-42ec-94ba-7696c3a1f3d5', metadata={'source': 'sp

In [13]:
## also we can pass directly vectors instead of query to get the similar documents

embedding_vector=embeddings.embed_query(query)
embedding_vector


[-0.4169941842556,
 2.0020790100097656,
 -3.0690503120422363,
 -1.1816940307617188,
 1.2064521312713623,
 1.1641331911087036,
 0.8199615478515625,
 1.083617925643921,
 0.12785014510154724,
 -0.04548805207014084,
 0.027160601690411568,
 2.0188698768615723,
 0.8003790974617004,
 1.501651644706726,
 1.567880392074585,
 -0.8116455078125,
 -0.5525689125061035,
 -0.6129799485206604,
 -0.990492045879364,
 1.8274933099746704,
 -0.9712417721748352,
 0.07770467549562454,
 -0.3749871253967285,
 0.22504368424415588,
 1.774756669998169,
 1.6832492351531982,
 -0.6403058171272278,
 0.303096204996109,
 -0.5852605700492859,
 0.10216464102268219,
 0.8406261205673218,
 -0.37980958819389343,
 -0.12047184258699417,
 -0.028379518538713455,
 -1.036152958869934,
 -1.3410784006118774,
 0.7427836060523987,
 1.1761469841003418,
 -0.30835017561912537,
 -0.6499677896499634,
 0.0818985253572464,
 -0.2204771637916565,
 -1.1240897178649902,
 -1.2815806865692139,
 2.304399251937866,
 -0.11344210803508759,
 0.150220036

In [14]:
db_score=db.similarity_search_by_vector(embedding_vector)
db_score

[Document(id='034f0cc7-6878-41bf-919c-ce663bdc17ae', metadata={'source': 'speech.txt'}, page_content='It is a distressing and oppressive duty, gentlemen of the Congress, which I have performed in thus addressing you. There are, it may be, many months of fiery trial and sacrifice ahead of us. It is a fearful thing to lead this great peaceful people into war, into the most terrible and disastrous of all wars, civilization itself seeming to be in the balance. But the right is more precious than peace, and we shall fight for the things which we have always carried nearest our heartsâ€”for democracy, for the right of those who submit to authority to have a voice in their own governments, for the rights and liberties of small nations, for a universal dominion of right by such a concert of free peoples as shall bring peace and safety to all nations and make the world itself at last free.'),
 Document(id='d0844a28-8c5b-42ec-94ba-7696c3a1f3d5', metadata={'source': 'speech.txt'}, page_content='T

In [17]:
### Saving and loading the FAISS database

'''The FAISS database can be saved to disk using the save_local method.'''
db.save_local("faiss_index")

In [18]:
new_db=FAISS.load_local("faiss_index",embeddings,allow_dangerous_deserialization=True)

docs=new_db.similarity_search(query)
docs[0].page_content

'It is a distressing and oppressive duty, gentlemen of the Congress, which I have performed in thus addressing you. There are, it may be, many months of fiery trial and sacrifice ahead of us. It is a fearful thing to lead this great peaceful people into war, into the most terrible and disastrous of all wars, civilization itself seeming to be in the balance. But the right is more precious than peace, and we shall fight for the things which we have always carried nearest our heartsâ€”for democracy, for the right of those who submit to authority to have a voice in their own governments, for the rights and liberties of small nations, for a universal dominion of right by such a concert of free peoples as shall bring peace and safety to all nations and make the world itself at last free.'